# Geometry-adaptive local neighborhood selection

In [ ]:
# Run the requirement file 
import numpy as np
import matplotlib.pyplot as plt

from scipy.spatial import cKDTree
from scipy.optimize import minimize
from sklearn.model_selection import train_test_split

from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from scipy.interpolate import griddata
import open3d as o3d
import numpy as np
import plotly.graph_objects as go
import potpourri3d as pp3d
from scipy.special import comb


## 0. Question and Problem

Given a 3D point cloud sampled from an underlying surface, the objective is to determine an appropriate local neighborhood size (K) around each query point.

A fixed (K) is not necessarily appropriate because the optimal neighborhood depends on:

1. local points density,
2. local curvature,
3. geometric features and boundaries.

The main question investigated in this notebook is:

**How can the local neighborhood size (K) be selected adaptively from the geometry of the point cloud?**

We compare two neighborhood-selection strategies (Error Estimation, one using statistical approach and the other one with geometric distance approach):

1. Variogram 
2. Average Neighborhood Erroe


## 1. General Problem Formulation

Let

$$
P = \{p_1,\ldots,p_N\}, \qquad p_i\in\mathbb{R}^3
$$

be a point cloud sampled from an underlying surface $\mathcal{S}$.

For a query point $q_i$, let

$$
N_i(K)
$$

denote its $K$-nearest-neighbor neighborhood.

Instead of fixing $K$ globally, we seek a **point-dependent neighborhood size**

$$
\boxed{K_i^*}
$$

such that the selected neighborhood is:

- sufficiently large for **stable local estimation**, while
- sufficiently small to **preserve the local geometry** of $\mathcal{S}$.

For each query point $q_i$, we consider a sequence of candidate neighborhood sizes

$$
K_{\min}
\rightarrow
K_{\min}+\Delta K
\rightarrow
K_{\min}+2\Delta K
\rightarrow \cdots \rightarrow
K_{\max}.
$$

At every scale $K$, a geometric criterion is evaluated to determine whether the current neighborhood remains locally valid.

The adaptive neighborhood selection can therefore be written as

$$
\boxed{
K_i^*
=
\operatorname*{arg\,max}_{K}
\left\{
K :
\text{local geometric criterion is satisfied}
\right\}.
}
$$

The criterion should balance the trade-off between **estimation stability** and **geometric locality**.

In this work, three criteria are considered:

1. **Quadratic surface approximation** — measures how well a local quadratic model represents the neighborhood.


The three approaches use the same candidate neighborhoods

$$
N_i(K_{\min}),\,
N_i(K_{\min}+\Delta K),\,
\ldots,\,
N_i(K_{\max}),
$$

so that the effect of the neighborhood-selection criterion can be compared independently.

## 2. Common pipeline

All these methods use the same initial procedure 
### Step 1 - Build a spatial search structure 
Construct a KD-tree from the input point-cloud 
We define the minimum number of points ($K_{min}$) = 5  for each neighborhood
### Step 2 - Select a query point
For each query point $q_i$, find the $K_{max}$ nearest neighbors
### Step 3 - Generate candidate neighborhoods
Define  $$K_m = K_{min} + m\Delta K$$ 
for  $m = 1,...,p $  and $\Delta K < K_{min}$
Generate  $N_i(K_m)= [p_{i,1}, ..., p_{i, K_m}]$
for $i$ corresponds to the number of neighborhood 

### Step 4 - Construct Bezier surface and Evaluate the local geometry criterion 
Explaination for each method 
1. Estimate an adaptive error tolerance from the local variogram of the normal displacements.
Why $r_i$ ?
The variogram model for this particular problem depends heavily on the displacement of the detrented normal $r_i$. The reason for this is because we want to measure the normal displacement for each Bezier patch computed from $N_i(K_m)$ of candidate $K_m$. This requires stationary data with no linear drift to accurately measure local surface roughness and noise.
The empirical variogram is
$$
\widehat{\gamma}_i(h)
=
\frac{1}{2|N_i(h)|}
\sum_{(j,k)\in N_i(h)}
(r_j-r_k)^2,
$$ 
where $r_j$ are normal-height residuals after removing a local linear trend.



### Step 5 - Select $K_{opt}$
The largest neighborhood satisfying the corresponding criterion is selected



## 3. Local Coordinate system 

Eventually, before applying any of these methods, the first step we need to do is compute the local coordinate system (i.e finding the tangent plan and compute )

In [ ]:
def local_coordinates (neighborhood):
    # Local PCA coordinate 
    centroid, normal, e1, e2, eigenvalues= local_pca(neighborhood)
    X= neighborhood - centroid 
    u = X@e1
    v= X@e2
    w=X@normal
    uvw= np.column_stack((u,v,w))
    return uvw,u,v,w
    

## 4. Method 1 : Local Bezier approximation

In [ ]:
def obtain_height(u, v, w):
    #compute the height 
    h = griddata(np.column_stack(u,v,w, method='linear'))
    nan_mask = np.isnan(h)
    if nan_mask.any():
        h_nn = griddata(np.column_stack(u,v,w, method='nearest'))
        h[nan_mask] = h_nn[nan_mask]
    return h

In [ ]:
# Fit Bezier Patch 
def bernstein(n, i, t):
    # Define the N, (degree+1)^2) Bernstein design matrix.
    return comb(n, i) * t**i * (1 - t) ** (n - i)


def bezier_design_matrix(s, t, degree):
    """s, t in [0,1], flattened. Returns (N, (degree+1)^2) Bernstein design matrix."""
    n = degree
    cols = []
    for i in range(n + 1):
        Bi = bernstein(n, i, s)
        for j in range(n + 1):
            Bj = bernstein(n, j, t)
            cols.append(Bi * Bj)
    return np.column_stack(cols)

def fit_bezier_patch(gu, gv, gh, degree=2):
    """
    Fit a degree x degree tensor-product Bezier patch to a height grid.
    Returns control_points with shape (degree+1, degree+1, 3) in (u, v, h),
    plus the (u,v) normalization bounds needed to evaluate the patch later.
    """
    umin, umax = gu.min(), gu.max()
    vmin, vmax = gv.min(), gv.max()
    s = (gu.ravel() - umin) / (umax - umin + 1e-12)   # normalize to [0,1]
    t = (gv.ravel() - vmin) / (vmax - vmin + 1e-12)

    A = bezier_design_matrix(s, t, degree)
    coeffs, *_ = np.linalg.lstsq(A, gh.ravel(), rcond=None)  # heights of control pts

    n = degree
    ctrl_h = coeffs.reshape(n + 1, n + 1)
    ctrl_u = umin + (umax - umin) * np.linspace(0, 1, n + 1)
    ctrl_v = vmin + (vmax - vmin) * np.linspace(0, 1, n + 1)
    CU, CV = np.meshgrid(ctrl_u, ctrl_v, indexing='ij')
    control_points = np.stack([CU, CV, ctrl_h], axis=-1)
    return control_points, (umin, umax, vmin, vmax)